In [ ]:
import os
import warnings
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
DATA_PATH       = ""   # e.g. "/kaggle/input/datasets/combined.csv"
METRICS_DIR     = ""   # e.g. "/kaggle/working/reports/metrics/lstm/"
RESULTS_DIR     = ""   # e.g. "/kaggle/working/data/meta_test/"
FIGURES_DIR     = ""   # e.g. "/kaggle/working/reports/figures/"
MODEL_DIR       = ""   # e.g. "/kaggle/working/models/lstm/"
OOF_DIR         = ""   # e.g. "/kaggle/working/data/meta_train/"
SUMMARY_METRICS = ""   # e.g. "/kaggle/working/reports/metrics/lstm/LSTM_all_tickers.csv"

SYMBOLS    = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

TRAIN_RATIO = 0.8
N_FOLDS     = 10
GAP_DAYS    = 10
LOOKBACK    = 7

HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.2
BATCH_SIZE  = 512   # large batch saturates both T4 GPUs, much faster than 32
EPOCHS      = 50
LR          = 1e-3
PATIENCE    = 7

# use both T4 GPUs with DataParallel; falls back to single GPU or CPU automatically
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4   # parallel dataloader workers, frees GPU from waiting on CPU

In [ ]:
def split_data(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [ ]:
def scale_data(train_df, test_df):
    scaler       = MinMaxScaler()
    train_df     = train_df.copy()
    test_df      = test_df.copy()
    train_df[OHLCV_COLS] = scaler.fit_transform(train_df[OHLCV_COLS])
    test_df[OHLCV_COLS]  = scaler.transform(test_df[OHLCV_COLS])
    close_scaler = MinMaxScaler()
    close_scaler.fit(train_df[["Close"]])
    return train_df, test_df, scaler, close_scaler

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, lookback: int):
        xs, ys = [], []
        for i in range(lookback, len(data)):
            xs.append(data[i - lookback:i])
            ys.append(data[i, 3])   # index 3 = Close
        # convert to tensor once at init so __getitem__ is a pure index op (no per-sample overhead)
        self.X = torch.tensor(np.array(xs), dtype=torch.float32)
        self.y = torch.tensor(np.array(ys), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(1)

In [ ]:
def build_loader(dataset, indices, shuffle):
    subset = torch.utils.data.Subset(dataset, indices)
    return DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,    # keeps tensors in pinned memory so GPU transfer is async and faster
        persistent_workers=True if NUM_WORKERS > 0 else False,  # reuse worker processes across epochs
    )

In [ ]:
def train_model(loader: DataLoader, input_size: int) -> nn.Module:
    base_model = LSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)

    # wrap with DataParallel to split each batch across both T4 GPUs automatically
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(base_model)
        log.info(f"      using {torch.cuda.device_count()} GPUs via DataParallel")
    else:
        model = base_model

    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scaler_amp = torch.cuda.amp.GradScaler()   # automatic mixed precision: fp16 compute, fp32 weights
    criterion = nn.MSELoss()

    best_loss  = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)  # non_blocking overlaps transfer with compute
            optimizer.zero_grad(set_to_none=True)   # set_to_none skips memset, slightly faster than zero_grad()
            with torch.cuda.amp.autocast():         # run forward in fp16 for speed
                pred = model(xb)
                loss = criterion(pred, yb)
            scaler_amp.scale(loss).backward()       # scale loss to avoid fp16 underflow
            scaler_amp.step(optimizer)
            scaler_amp.update()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)

        if avg_loss < best_loss:
            best_loss  = avg_loss
            # save state from base_model (unwrap DataParallel) so we can reload without DataParallel later
            src = model.module if isinstance(model, nn.DataParallel) else model
            best_state = {k: v.cpu().clone() for k, v in src.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                log.info(f"      early stopping at epoch {epoch+1}, best_loss={best_loss:.6f}")
                break

    base_model.load_state_dict(best_state)
    return base_model   # always return the unwrapped base model

In [ ]:
def predict_sequences(model: nn.Module, data: np.ndarray, lookback: int) -> np.ndarray:
    model.eval()
    dataset = TimeSeriesDataset(data, lookback)
    loader  = DataLoader(
        dataset,
        batch_size=BATCH_SIZE * 4,   # larger batch at inference since no gradient storage needed
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast():
                pred = model(xb)
            preds.extend(pred.cpu().numpy())
    return np.array(preds)

In [ ]:
def run_oof(train_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    n         = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)
    data      = train_df[OHLCV_COLS].values

    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}, lookback={LOOKBACK}")

    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS
        val_end   = (k + 1) * fold_size if k < N_FOLDS else n

        if val_start + LOOKBACK >= val_end:
            log.warning(f"      Fold {k}: skipped — val window too small")
            continue

        fold_data     = data[:val_end]
        dataset       = TimeSeriesDataset(fold_data, LOOKBACK)
        train_samples = [i for i in range(len(dataset)) if (i + LOOKBACK) < train_end]
        val_samples   = [i for i in range(len(dataset)) if val_start <= (i + LOOKBACK) < val_end]

        if not train_samples or not val_samples:
            continue

        train_loader = build_loader(dataset, train_samples, shuffle=True)

        try:
            model  = train_model(train_loader, input_size=len(OHLCV_COLS))
            val_data = fold_data[val_start - LOOKBACK:val_end]
            preds    = predict_sequences(model, val_data, LOOKBACK)

            actual_val = train_df["Close"].values[val_start:val_end]
            oof_preds[val_start:val_start + len(preds)] = preds

            mae = np.mean(np.abs(preds - actual_val[:len(preds)]))
            log.info(f"      Fold {k}: train=[0:{train_end}]  val=[{val_start}:{val_end}]  MAE={mae:.6f}")

            # free GPU memory between folds so next fold starts with a clean slate
            del model
            torch.cuda.empty_cache()

        except Exception as e:
            log.warning(f"      Fold {k} failed: {e}")
            oof_preds[val_start:val_end] = np.nan

    nan_count = np.isnan(oof_preds).sum()
    print(f"  oof nan: {nan_count}/{n} ({nan_count/n:.1%}) — first ~{fold_size + GAP_DAYS + LOOKBACK} rows uncovered")

    return pd.DataFrame({
        "Ticker":               ticker,
        "Date":                 train_df["Date"].values,
        "Close":                train_df["Close"].values,
        "LSTM_Predicted_Close": oof_preds,
    })

In [ ]:
def predict_test(model, train_df: pd.DataFrame, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    context   = train_df[OHLCV_COLS].values[-LOOKBACK:]
    test_data = np.concatenate([context, test_df[OHLCV_COLS].values], axis=0)
    preds     = predict_sequences(model, test_data, LOOKBACK)
    return pd.DataFrame({
        "Ticker":               ticker,
        "Date":                 test_df["Date"].values,
        "Close":                test_df["Close"].values,
        "LSTM_Predicted_Close": preds,
    })

In [ ]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "LSTM_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true + 1e-8))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)

    mask  = actual_dir != 0
    tpa   = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE":    mse,
        "MAE":    mae,
        "MAPE":   mape,
        "RMSE":   rmse,
        "R2":     r2,
        "DA":     da,
        "TPA":    tpa,
        "V-RMSE": v_rmse,
    }])

In [ ]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=["LSTM_Predicted_Close"])

    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    axes[0].plot(df_plot["Date"], df_plot["Close"],
                 label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot["LSTM_Predicted_Close"],
                 label="LSTM Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"LSTM predicted results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    error  = df_plot["Close"] - df_plot["LSTM_Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [ ]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    train_df, test_df = split_data(df)
    train_df, test_df, scaler, close_scaler = scale_data(train_df, test_df)
    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")

    oof_df = run_oof(train_df, ticker)

    log.info("    Training final model on full train set...")
    full_dataset = TimeSeriesDataset(train_df[OHLCV_COLS].values, LOOKBACK)
    full_loader  = build_loader(full_dataset, list(range(len(full_dataset))), shuffle=True)
    final_model  = train_model(full_loader, input_size=len(OHLCV_COLS))

    test_result_df = predict_test(final_model, train_df, test_df, ticker)

    def inverse_close(df_in, col):
        df_in = df_in.copy()
        mask  = df_in[col].notna()
        df_in.loc[mask, [col]] = close_scaler.inverse_transform(df_in.loc[mask, [col]])
        df_in[["Close"]]       = close_scaler.inverse_transform(df_in[["Close"]])
        return df_in

    oof_df         = inverse_close(oof_df, "LSTM_Predicted_Close")
    test_result_df = inverse_close(test_result_df, "LSTM_Predicted_Close")

    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)
    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.2f} "
             f"DA={metrics_df['DA'].iloc[0]:.2%}  "
             f"R2={metrics_df['R2'].iloc[0]:.4f}")

    test_metrics = calc_metrics(test_result_df)
    test_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.2f} "
             f"DA={test_metrics['DA'].iloc[0]:.2%}  "
             f"R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(os.path.join(METRICS_DIR, f"LSTM_{ticker}_metrics.csv"), index=False)

    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(os.path.join(OOF_DIR, f"oof_lstm_{ticker}.csv"), index=False)

    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_result_df.to_csv(os.path.join(RESULTS_DIR, f"LSTM_{ticker}_predictions.csv"), index=False)

    plot_results(test_result_df, save_dir=FIGURES_DIR)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"lstm_{ticker}.pkl"), "wb") as f:
        pickle.dump({"model_state": final_model.state_dict(), "scaler": scaler,
                     "close_scaler": close_scaler, "ticker": ticker}, f)

    torch.cuda.empty_cache()
    log.info(f"  All outputs saved for {ticker}")
    return test_metrics

In [ ]:
all_metrics = []

for ticker in SYMBOLS:
    try:
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"\nSummary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")